<a href="https://colab.research.google.com/github/IrisCheon/NLP-practice/blob/main/News_Headlines_Topic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# News Headline Topic Modeling

This project explores topic modeling on short news headlines using LDA and NMF. Rather than producing clearly separated topics, the models showed limited interpretability, suggesting that short and sparse headline texts can make topic discovery difficult.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("therohk/million-headlines")

print("Path to dataset files:", path)

100%|██████████| 21.4M/21.4M [00:01<00:00, 14.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/therohk/million-headlines/versions/5


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [3]:
print(os.listdir(path))

['abcnews-date-text.csv']


In [6]:
df = pd.read_csv(os.path.join(path, 'abcnews-date-text.csv'))
df.head()

,publish_date,headline_text
0,20030219,aba decides against community broadcasting lic...
1,20030219,act fire witnesses must be aware of defamation
2,20030219,a g calls for infrastructure protection summit
3,20030219,air nz staff in aust strike for pay rise
4,20030219,air nz strike to affect australian travellers


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1244184 entries, 0 to 1244183
Data columns (total 2 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   publish_date   1244184 non-null  int64 
 1   headline_text  1244184 non-null  object
dtypes: int64(1), object(1)
memory usage: 19.0+ MB


In [9]:
print(df["publish_date"].min(), df["publish_date"].max())

20030219 20211231


In [17]:
for i in range(10):
    print("="*30)
    print(
        df["headline_text"][i],
        end= '\n'
    )

aba decides against community broadcasting licence
act fire witnesses must be aware of defamation
a g calls for infrastructure protection summit
air nz staff in aust strike for pay rise
air nz strike to affect australian travellers
ambitious olsson wins triple jump
antic delighted with record breaking barca
aussie qualifier stosur wastes four memphis match
aust addresses un security council over iraq
australia is locked into war timetable opp


In [18]:
sample_df = df.sample(30000, random_state = 42)

# ■ LDA

In [23]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
            stop_words = "english",
            max_df = 0.95,
            min_df = 5,
            max_features = 5000)

X_counts = vectorizer.fit_transform(sample_df["headline_text"])

In [21]:
n_topics = 8

In [26]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(
    n_components = n_topics,
    random_state = 42,
    learning_method = "batch",
    max_iter = 20
)

lda.fit(X_counts)

LatentDirichletAllocation(max_iter=20, n_components=8, random_state=42)

## ■ Topic별 대표 단어

In [27]:
feature_names = vectorizer.get_feature_names_out()

In [28]:
def display_topics(model, feature_names, n_top_words = 10):
    for topic_idx, topic in enumerate(model.components_):
        top_word_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_word_indices]
        print(f"Topic {topic_idx}:")
        print(", ".join(top_words))
        print()

In [29]:
display_topics(lda, feature_names, 15)

Topic 0:
police, man, charged, murder, court, accused, drug, woman, charges, australian, death, probe, test, australia, arrested

Topic 1:
water, qld, trial, govt, rise, urged, country, day, budget, years, council, record, canberra, help, new

Topic 2:
world, cup, new, adelaide, centre, hospital, time, sydney, covid, close, jailed, league, melbourne, attack, 19

Topic 3:
interview, farmers, australia, missing, search, run, driver, hit, family, park, drought, new, home, pm, pacific

Topic 4:
government, election, south, north, says, new, coast, talks, west, win, wins, gold, weather, assault, east

Topic 5:
ban, court, says, death, case, future, market, tasmania, inquiry, calls, council, workers, told, job, australian

Topic 6:
health, rural, govt, news, change, sa, national, defends, road, plan, group, campaign, nsw, abc, guilty

Topic 7:
crash, report, dies, school, indigenous, killed, nsw, man, car, dead, calls, plans, jobs, review, rail



In [ ]:
"""
Topic 0:
police, man, charged, murder, court, accused, drug, woman, charges, australian, death, probe, test, australia, arrested
    > Crime

Topic 1:
water, qld, trial, govt, rise, urged, country, day, budget, years, council, record, canberra, help, new
    > Economy

Topic 2:
world, cup, new, adelaide, centre, hospital, time, sydney, covid, close, jailed, league, melbourne, attack, 19
    > Sports?

Topic 3:
interview, farmers, australia, missing, search, run, driver, hit, family, park, drought, new, home, pm, pacific
    > Agriculture?

Topic 4:
government, election, south, north, says, new, coast, talks, west, win, wins, gold, weather, assault, east
    > Politics

Topic 5:
ban, court, says, death, case, future, market, tasmania, inquiry, calls, council, workers, told, job, australian
    > Crime?

Topic 6:
health, rural, govt, news, change, sa, national, defends, road, plan, group, campaign, nsw, abc, guilty
    > Health?

Topic 7:
crash, report, dies, school, indigenous, killed, nsw, man, car, dead, calls, plans, jobs, review, rail
    > Crime?

"""

In [30]:
doc_topic_dist = lda.transform(X_counts)
doc_topic_df = pd.DataFrame(
    doc_topic_dist,
    columns = [f"Topic {i}" for i in range(n_topics)]
)

doc_topic_df.head()

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7
0,0.015638,0.173185,0.015634,0.015650,0.419397,0.015629,0.329234,0.015633
1,0.031250,0.514907,0.031254,0.297482,0.031288,0.031250,0.031319,0.031250
2,0.031250,0.781239,0.031250,0.031250,0.031250,0.031255,0.031256,0.031250
3,0.031250,0.031250,0.031250,0.281239,0.031250,0.031250,0.531261,0.031250
4,0.031297,0.031250,0.031295,0.031250,0.531167,0.031250,0.031250,0.281242


In [31]:
topic_cols = [f"Topic {i}" for i in range (n_topics)]

doc_topic_df["dominant_topic"] = doc_topic_df[topic_cols].idxmax(axis=1)
doc_topic_df["dominant_topic_score"] = doc_topic_df[topic_cols].max(axis=1)

doc_topic_df.sort_values("dominant_topic_score", ascending = False)

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,dominant_topic,dominant_topic_score
24477,0.012505,0.012501,0.012508,0.012519,0.912459,0.012500,0.012500,0.012507,Topic 4,0.912459
23315,0.012515,0.012502,0.912455,0.012510,0.012507,0.012504,0.012509,0.012500,Topic 2,0.912455
13220,0.012538,0.012503,0.012511,0.912411,0.012515,0.012503,0.012519,0.012501,Topic 3,0.912411
18405,0.013892,0.013891,0.902768,0.013889,0.013893,0.013889,0.013889,0.013889,Topic 2,0.902768
7020,0.902761,0.013891,0.013889,0.013890,0.013896,0.013891,0.013890,0.013892,Topic 0,0.902761
...,...,...,...,...,...,...,...,...,...,...
13115,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000
21650,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000
17409,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000
117,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000


In [32]:
doc_topic_df["dominant_topic_score"].mean()
# 확신이 57%밖에 안 됨 → 모델을 재 설정 해보자

np.float64(0.5789355697175614)

## ■ 모델 재설정(binary)

In [38]:
vectorizer2 = CountVectorizer(
            stop_words = "english",
            ngram_range = (1,2),
            max_df = 0.95,
            min_df = 5,
            max_features = 5000)

X_counts2 = vectorizer2.fit_transform(sample_df["headline_text"])

In [40]:
lda2 = LatentDirichletAllocation(
    n_components = n_topics,
    random_state = 42,
    learning_method = "batch",
    max_iter = 20
)

lda2.fit(X_counts2)

LatentDirichletAllocation(max_iter=20, n_components=8, random_state=42)

In [43]:
feature_names2 = vectorizer2.get_feature_names_out()

def display_topics2(model, feature_names2, n_top_words = 10):
    for topic_idx, topic in enumerate(model.components_):
        top_word_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names2[i] for i in top_word_indices]
        print(f"Topic {topic_idx}:")
        print(", ".join(top_words))
        print()

In [44]:
display_topics2(lda2, feature_names2, 15)

Topic 0:
report, day, dead, killed, crash, new, man, change, car, dies, hospital, future, injured, test, brisbane

Topic 1:
australian, australia, national, open, residents, win, canberra, family, rural, india, season, talks, continues, sale, race

Topic 2:
world, cup, abc, country, record, wins, news, trump, win, says, world cup, hour, tour, england, country hour

Topic 3:
police, man, court, death, accused, woman, sydney, pm, charged, murder, charges, drug, probe, assault, jailed

Topic 4:
govt, new, plan, gold, sex, ban, indigenous, urged, coast, opposition, mayor, bid, calls, centre, mp

Topic 5:
interview, sa, says, police, market, coronavirus, west, covid, fatal, south, wa, price, war, rates, run

Topic 6:
health, election, queensland, minister, says, study, case, state, arrested, new, claims, murder, charged, labor, cancer

Topic 7:
water, new, council, nsw, boost, government, power, high, group, funding, plan, workers, changes, fears, plans



In [45]:
doc_topic_dist2 = lda2.transform(X_counts2)
doc_topic_df2 = pd.DataFrame(
    doc_topic_dist2,
    columns = [f"Topic {i}" for i in range(n_topics)]
)

doc_topic_df2.head()

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7
0,0.209368,0.177707,0.013890,0.013889,0.013898,0.013910,0.296609,0.260730
1,0.031250,0.031276,0.031298,0.031250,0.031257,0.031271,0.325211,0.487187
2,0.281244,0.031250,0.531229,0.031267,0.031250,0.031260,0.031250,0.031250
3,0.031250,0.031288,0.250676,0.031250,0.031250,0.561786,0.031250,0.031250
4,0.781136,0.031261,0.031258,0.031250,0.031250,0.031250,0.031264,0.031330


In [46]:
topic_cols = [f"Topic {i}" for i in range (n_topics)]

doc_topic_df2["dominant_topic"] = doc_topic_df2[topic_cols].idxmax(axis=1)
doc_topic_df2["dominant_topic_score"] = doc_topic_df2[topic_cols].max(axis=1)

doc_topic_df2.sort_values("dominant_topic_score", ascending = False)

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,dominant_topic,dominant_topic_score
24414,0.008335,0.008348,0.008364,0.008334,0.008334,0.941612,0.008336,0.008337,Topic 5,0.941612
13525,0.008339,0.008348,0.008365,0.008334,0.008337,0.941598,0.008340,0.008339,Topic 5,0.941598
14496,0.937478,0.008936,0.008931,0.008930,0.008929,0.008936,0.008931,0.008930,Topic 0,0.937478
91,0.009615,0.009616,0.932686,0.009615,0.009616,0.009620,0.009616,0.009616,Topic 2,0.932686
23315,0.932613,0.009639,0.009630,0.009616,0.009616,0.009627,0.009616,0.009643,Topic 0,0.932613
...,...,...,...,...,...,...,...,...,...,...
6533,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000
23569,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000
29544,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000
1482,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,Topic 0,0.125000


In [48]:
doc_topic_df2["dominant_topic_score"].mean()
# 살짝 올랐지만... 유의미하지 않음.

np.float64(0.5878867922877042)

## ■ 모델 재설정(n_topics = 5)

In [50]:
n_topics=5

lda = LatentDirichletAllocation(
    n_components = n_topics,
    random_state = 42,
    learning_method = "batch",
    max_iter = 20
)

lda.fit(X_counts)   # 기존 unigram으로 다시

LatentDirichletAllocation(max_iter=20, n_components=5, random_state=42)

In [51]:
display_topics(lda, feature_names, 15)

Topic 0:
police, man, court, crash, charged, murder, woman, death, car, accused, rural, dies, australian, drug, market

Topic 1:
govt, health, water, council, world, new, plan, says, high, group, calls, nsw, claims, country, change

Topic 2:
new, hospital, attack, centre, coronavirus, says, killed, time, covid, injured, adelaide, dead, people, housing, power

Topic 3:
interview, sydney, minister, trial, sex, child, pm, australia, guilty, drought, assault, missing, family, home, abuse

Topic 4:
says, coast, win, south, north, election, new, government, talks, abc, gold, west, wins, weather, nsw



In [ ]:
"""
Topic 0:
police, man, court, crash, charged, murder, woman, death, car, accused, rural, dies, australian, drug, market
    > Crime

Topic 1:
govt, health, water, council, world, new, plan, says, high, group, calls, nsw, claims, country, change
    > Health?

Topic 2:
new, hospital, attack, centre, coronavirus, says, killed, time, covid, injured, adelaide, dead, people, housing, power
    > Health??

Topic 3:
interview, sydney, minister, trial, sex, child, pm, australia, guilty, drought, assault, missing, family, home, abuse
    > Crime?

Topic 4:
says, coast, win, south, north, election, new, government, talks, abc, gold, west, wins, weather, nsw
    > Politics
"""

※ LDA 모델은 unigram, bigram, n_topics 조절해도 모두 명확하게 topic이 잡히지 않음
- headline이라는 데이터의 특성상, 참고할 수 있는 단어 수 자체가 적어 발생한 문제로 확인

LDA with both unigram and bigram features produced relatively ambiguous topics. Reducing the number of topics did not substantially improve interpretability. This suggests that the issue may come from the short and sparse nature of news headlines rather than only from hyperparameter choice.

# ■ TfidVectorizer + NMF

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [53]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    max_df=0.8,
    min_df=10,
    max_features=5000,
    ngram_range=(1, 2)
)

X_tfidf = tfidf_vectorizer.fit_transform(sample_df["headline_text"])

In [54]:
nmf = NMF(
    n_components=5,
    random_state=42,
    max_iter=300
)

W = nmf.fit_transform(X_tfidf)  # 각 문서가 각 topic을 얼마나 가지는가
H = nmf.components_         # 각 topic이 어떤 단어를 강하게 가지는가

## ■ Topic별 대표 단어

In [56]:
feature_names = tfidf_vectorizer.get_feature_names_out()

def display_topics(model, feature_names, n_top_words = 15):
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_indices]

        print(f"\nTopic {topic_idx}")
        print(", ".join(top_words))

display_topics(nmf, feature_names, n_top_words = 15)


Topic 0
interview, extended interview, extended, andrew, tim, interview david, ben, matt, interview john, james, david, john, nrl interview, luke, cameron

Topic 1
police, investigate, police investigate, probe, search, missing, police probe, car, arrest, police search, officer, death, woman, police arrest, police officer

Topic 2
new, hospital, laws, set, zealand, new zealand, australia, year, york, deal, covid, records, new york, chief, cases

Topic 3
man, charged, court, man charged, murder, accused, crash, dies, car, woman, jailed, death, man dies, killed, man jailed

Topic 4
says, council, govt, nsw, australia, plan, water, health, report, qld, wa, rural, election, government, calls


In [ ]:
'''
Topic 0
interview, extended interview, extended, andrew, tim, interview david, ben, matt, interview john, james, david, john, nrl interview, luke, cameron
    > Interview?

Topic 1
police, investigate, police investigate, probe, search, missing, police probe, car, arrest, police search, officer, death, woman, police arrest, police officer
    > Crime

Topic 2
new, hospital, laws, set, zealand, new zealand, australia, year, york, deal, covid, records, new york, chief, cases
    > Health

Topic 3
man, charged, court, man charged, murder, accused, crash, dies, car, woman, jailed, death, man dies, killed, man jailed
    > Crime

Topic 4
says, council, govt, nsw, australia, plan, water, health, report, qld, wa, rural, election, government, calls
    > Politics? or Environment?
'''

## ■ 문서별 Topic 비율 확인

In [57]:
doc_topic_df = pd.DataFrame(
    W,
    columns=[f"Topic {i}" for i in range(n_topics)]
)

doc_topic_df.head()

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4
0,0.000000,0.000000,0.006633,0.002936,0.028802
1,0.000000,0.000096,0.000460,0.000000,0.021011
2,0.000009,0.000128,0.000561,0.000914,0.001806
3,0.000000,0.000267,0.000000,0.000000,0.009761
4,0.000228,0.000708,0.001856,0.000089,0.005629


In [63]:
doc_topic_ratio = W / (W.sum(axis=1, keepdims=True) + 1e-10)
                                        # shape 유지용 / 아주 작은 수(zerodivision 방지)
doc_topic_ratio_df = pd.DataFrame(
    doc_topic_ratio,
    columns=[f"Topic {i}" for i in range(n_topics)]
)

doc_topic_ratio_df.head()

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4
0,0.000000,0.000000,0.172864,0.076514,0.750622
1,0.000000,0.004459,0.021323,0.000000,0.974218
2,0.002748,0.037347,0.164020,0.267440,0.528444
3,0.000000,0.026595,0.000000,0.000000,0.973405
4,0.026767,0.083192,0.218147,0.010436,0.661458


In [67]:
columns=[f"Topic {i}" for i in range(n_topics)]

doc_topic_ratio_df["dominant_topic"] = doc_topic_ratio_df[columns].idxmax(axis=1)
doc_topic_ratio_df["dominant_score"] = doc_topic_ratio_df[columns].max(axis=1)

doc_topic_ratio_df.describe()

,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,dominant_score
count,30000.0000,30000.0000,30000.0000,30000.0000,30000.0000,30000.0000
mean,0.0244,0.0930,0.1133,0.1607,0.5973,0.7454
std,0.1227,0.1785,0.1746,0.2501,0.3211,0.2039
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.0000,0.0000,0.0000,0.0000,0.3728,0.5953
50%,0.0000,0.0286,0.0582,0.0418,0.6747,0.7734
75%,0.0000,0.1065,0.1484,0.2070,0.8681,0.9209
max,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


## ■ 실제 데이터 확인

In [68]:
result_df = sample_df.reset_index(drop=True).copy()

result_df = pd.concat(
    [result_df, doc_topic_ratio_df],
    axis=1
)

In [71]:
def show_representative_headlines(df, topic_name, n=10):
    topic_docs = df.sort_values(by=topic_name, ascending=False).head(n)

    for idx, row in topic_docs.iterrows():
        print(topic_name)
        print(f"[{idx}] score={row[topic_name]:.3f}")
        print(row["headline_text"])
        print()

In [72]:
for i in range(5):
    print(show_representative_headlines(result_df, f"Topic {i}", n=5))

Topic 0
[19814] score=1.000
interview caitlin bassett

Topic 0
[19757] score=1.000
marty mattner interview

Topic 0
[23733] score=1.000
interview pat rafter

Topic 0
[23772] score=1.000
interview jamie morgan

Topic 0
[25839] score=1.000
interview madison browne

None
Topic 1
[6460] score=1.000
della bosca to be interviewed by police

Topic 1
[20483] score=1.000
police thresher

Topic 1
[27236] score=1.000
police to scale back cronulla patrols

Topic 1
[14210] score=1.000
police take possession of bone found on clifton

Topic 1
[7923] score=1.000
police put brakes on speedsters

None
Topic 2
[9749] score=1.000
hillary silences bill over new bosnia gaffe

Topic 2
[28672] score=1.000
akainacephalus johnsoni new armoured dinosaur unearthed

Topic 2
[7079] score=1.000
new lifeline for 8ccc

Topic 2
[27003] score=1.000
engineers collaborate with rivals in new silicon

Topic 2
[2863] score=1.000
new piaf film manager woos new somerville audiences

None
Topic 3
[22256] score=1.000
man crushed

※ 결과분석
- Topic 0 : 'interview'

- Topic 1 : 'police'

- Topic 2 : 'new'

- Topic 3 : 'man'

- Topic 4 : 'say'

전반적으로 특정 단어만 강하게 탐지한 것으로 보임


In [75]:
sample_df["headline_text"].str.split().str.len().describe()

,headline_text
count,30000.0000
mean,6.5549
std,1.8787
min,1.0000
25%,5.0000
50%,7.0000
75%,8.0000
max,14.0000


※ 헤드라인의 평균 단어 수가 6개, 많아도 14개이므로 분류에 한계가 있어 보임

# ■ Conclusion

The topic modeling results were not sufficiently interpretable. Both LDA and NMF tended to form topics around a small number of highly influential words rather than broader semantic themes. A likely reason is the short length of the headlines: the average headline contained only around six words, and most headlines contained fewer than eight words. Because topic modeling relies heavily on word co-occurrence patterns, the corpus did not provide enough contextual information for stable topic discovery.

Therefore, further hyperparameter tuning was considered less meaningful. This experiment suggests that headline-only data may be more suitable for temporal keyword analysis or supervised classification than for traditional topic modeling.